In [2]:
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)

In [35]:
df = pd.read_excel("../data/raw/telco_customer_churn.xlsx", engine="openpyxl", decimal=",")


In [8]:
df.shape

(7043, 33)

In [6]:
df.columns

Index(['CustomerID', 'Count', 'Country', 'State', 'City', 'Zip Code',
       'Lat Long', 'Latitude', 'Longitude', 'Gender', 'Senior Citizen',
       'Partner', 'Dependents', 'Tenure Months', 'Phone Service',
       'Multiple Lines', 'Internet Service', 'Online Security',
       'Online Backup', 'Device Protection', 'Tech Support', 'Streaming TV',
       'Streaming Movies', 'Contract', 'Paperless Billing', 'Payment Method',
       'Monthly Charges', 'Total Charges', 'Churn Label', 'Churn Value',
       'Churn Score', 'CLTV', 'Churn Reason'],
      dtype='str')

In [34]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
Count,7043.0,1.000000,0.000000,1.000000,1.000000,1.000000,1.000000,1.000000
Zip Code,7043.0,93521.964646,1865.794555,90001.000000,92102.000000,93552.000000,95351.000000,96161.000000
Latitude,7043.0,36.282441,2.455723,32.555828,34.030915,36.391777,38.224869,41.962127
Longitude,7043.0,-119.798880,2.157889,-124.301372,-121.815412,-119.730885,-118.043237,-114.192901
Tenure Months,7043.0,32.371149,24.559481,0.000000,9.000000,29.000000,55.000000,72.000000
Monthly Charges,7043.0,64.761692,30.090047,18.250000,35.500000,70.350000,89.850000,118.750000
Churn Value,7043.0,0.265370,0.441561,0.000000,0.000000,0.000000,1.000000,1.000000
Churn Score,7043.0,58.699418,21.525131,5.000000,40.000000,61.000000,75.000000,100.000000
CLTV,7043.0,4400.295755,1183.057152,2003.000000,3469.000000,4527.000000,5380.500000,6500.000000


In [35]:
df.head()

,CustomerID,Count,Country,State,City,Zip Code,Lat Long,Latitude,Longitude,Gender,Senior Citizen,Partner,Dependents,Tenure Months,Phone Service,Multiple Lines,Internet Service,Online Security,Online Backup,Device Protection,Tech Support,Streaming TV,Streaming Movies,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges,Churn Label,Churn Value,Churn Score,CLTV,Churn Reason
0,3668-QPYBK,1,United States,California,Los Angeles,90003,"33.964131, -118.272783",33.964131,-118.272783,Male,No,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,1,86,3239,Competitor made better offer
1,9237-HQITU,1,United States,California,Los Angeles,90005,"34.059281, -118.30742",34.059281,-118.307420,Female,No,No,Yes,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,1,67,2701,Moved
2,9305-CDSKC,1,United States,California,Los Angeles,90006,"34.048013, -118.293953",34.048013,-118.293953,Female,No,No,Yes,8,Yes,Yes,Fiber optic,No,No,Yes,No,Yes,Yes,Month-to-month,Yes,Electronic check,99.65,820.5,Yes,1,86,5372,Moved
3,7892-POOKP,1,United States,California,Los Angeles,90010,"34.062125, -118.315709",34.062125,-118.315709,Female,No,Yes,Yes,28,Yes,Yes,Fiber optic,No,No,Yes,Yes,Yes,Yes,Month-to-month,Yes,Electronic check,104.80,3046.05,Yes,1,84,5003,Moved
4,0280-XJGEX,1,United States,California,Los Angeles,90015,"34.039224, -118.266293",34.039224,-118.266293,Male,No,No,Yes,49,Yes,Yes,Fiber optic,No,Yes,Yes,No,Yes,Yes,Month-to-month,Yes,Bank transfer (automatic),103.70,5036.3,Yes,1,89,5340,Competitor had better devices


In [36]:
df.dtypes

CustomerID               str
Count                  int64
Country                  str
State                    str
City                     str
Zip Code               int64
Lat Long                 str
Latitude             float64
Longitude            float64
Gender                   str
Senior Citizen           str
Partner                  str
Dependents               str
Tenure Months          int64
Phone Service            str
Multiple Lines           str
Internet Service         str
Online Security          str
Online Backup            str
Device Protection        str
Tech Support             str
Streaming TV             str
Streaming Movies         str
Contract                 str
Paperless Billing        str
Payment Method           str
Monthly Charges      float64
Total Charges         object
Churn Label              str
Churn Value            int64
Churn Score            int64
CLTV                   int64
Churn Reason             str
dtype: object

In [27]:
pd.DataFrame({
    "dtype": df.dtypes,
    "non_null": df.count(),
    "nulls": df.isnull().sum(),
    "duplicates": df.duplicated().sum()
})

,dtype,non_null,nulls,duplicates
CustomerID,object,7043,0,0
Count,int64,7043,0,0
Country,object,7043,0,0
State,object,7043,0,0
City,object,7043,0,0
Zip Code,int64,7043,0,0
Lat Long,object,7043,0,0
Latitude,float64,7043,0,0
Longitude,float64,7043,0,0
Gender,object,7043,0,0


In [36]:
df["Total Charges"] = pd.to_numeric(
    df["Total Charges"]
        .astype(str)
        .str.replace(",", ".")
        .str.strip(),
    errors="coerce"
)

In [37]:
df = df.dropna(subset=["Total Charges"])

In [38]:
customers = df[[
    "CustomerID",
    "Gender",
    "Senior Citizen",
    "Partner",
    "Dependents",
    "Tenure Months",
    "Country",
    "State",
    "City",
    "Zip Code",
    "Latitude",
    "Longitude",
]].copy()

services = df[
    [
        "CustomerID",
        "Phone Service",
        "Multiple Lines",
        "Internet Service",
        "Online Security",
        "Online Backup",
        "Device Protection",
        "Tech Support",
        "Streaming TV",
        "Streaming Movies",
        "Contract",
        "Paperless Billing",
        "Payment Method",
        "Monthly Charges",
        "Total Charges",
    ]
].copy()

churn = df[
    [
        "CustomerID",
        "Churn Label",
        "Churn Value",
        "Churn Score",
        "CLTV",
        "Churn Reason",
    ]
].copy()

In [39]:
customers.columns = [
    "customer_id",
    "gender",
    "senior_citizen",
    "partner",
    "dependents",
    "tenure_months",
    "country",
    "state",
    "city",
    "zip_code",
    "latitude",
    "longitude",
]

services.columns = [
    "customer_id",
    "phone_service",
    "multiple_lines",
    "internet_service",
    "online_security",
    "online_backup",
    "device_protection",
    "tech_support",
    "streaming_tv",
    "streaming_movies",
    "contract_type",
    "paperless_billing",
    "payment_method",
    "monthly_charges",
    "total_charges",
]

churn.columns = [
    "customer_id",
    "churn_label",
    "churn_value",
    "churn_score",
    "cltv",
    "churn_reason",
]

In [151]:
bool_cols_customers = ["senior_citizen", "partner", "dependents"]


for col in bool_cols_customers:
    customers[col] = customers[col].astype(str).str.strip().map({"Yes": 1, "No": 0})


In [45]:
services.head()

,customer_id,phone_service,multiple_lines,internet_service,online_security,online_backup,device_protection,tech_support,streaming_tv,streaming_movies,contract_type,paperless_billing,payment_method,monthly_charges,total_charges
0,3668-QPYBK,1,0,DSL,1,1,0,0,0,0,Month-to-month,1,Mailed check,53.85,108.15
1,9237-HQITU,1,0,Fiber optic,0,0,0,0,0,0,Month-to-month,1,Electronic check,70.70,151.65
2,9305-CDSKC,1,1,Fiber optic,0,0,1,0,1,1,Month-to-month,1,Electronic check,99.65,820.50
3,7892-POOKP,1,1,Fiber optic,0,0,1,1,1,1,Month-to-month,1,Electronic check,104.80,3046.05
4,0280-XJGEX,1,1,Fiber optic,0,1,1,0,1,1,Month-to-month,1,Bank transfer (automatic),103.70,5036.30


In [43]:
services['tech_support'].unique()

array(['No', 'Yes', 'No internet service'], dtype=object)

In [44]:
bool_cols_services = [
    "phone_service",
    "multiple_lines",
    "online_security",
    "online_backup",
    "device_protection",
    "tech_support",
    "streaming_tv",
    "streaming_movies",
    "paperless_billing",
]

mapping = {
    "Yes": 1,
    "No": 0,
    "No internet service": 0,
    "No phone service": 0
}

for col in bool_cols_services:
    services[col] = services[col].astype(str).str.strip().map(mapping).astype("Int64")




In [149]:

churn["churn_label"] = churn["churn_label"].astype(str).str.strip().map({"Yes": 1, "No": 0})

In [ ]:
print(customers.shape)
print(services.shape)
print(churn.shape)

print(customers["customer_id"].nunique(), len(customers))
print(services["customer_id"].nunique(), len(services))
print(churn["customer_id"].nunique(), len(churn))

(7032, 12)
(7032, 15)
(7032, 6)
7032 7032
7032 7032
7032 7032


In [ ]:
customers.to_csv("../data/processed/customers.csv", index=False)
services.to_csv("../data/processed/services.csv", index=False)
churn.to_csv("../data/processed/churn.csv", index=False)
